In [0]:
from dbx_hybrid_search import hybrid_search_rrf
import openai
from pyspark.sql import functions as F
from LLMs_call import load_api_key, ask_gpt, build_messages, dbx_llm_chat
import markdown
from tqdm import tqdm
import mlflow

In [0]:
def display_MD(md):
    displayHTML(markdown.markdown(md, extensions=["fenced_code", "tables"]))

In [0]:
def RAG_pipe_main(spark, query, LLM_model, use_dbx_model=True, temperature=None, top_each=100, top_n=20, rrf_k=20, return_with_text=True):

    search_df = hybrid_search_rrf(spark, query, top_each=top_each, top_n=top_n, rrf_k=rrf_k, return_with_text=return_with_text)
    search_text_collection = search_df.orderBy(F.col("rrf_score").desc()).select(F.col("text")).limit(10).collect()
    messages = build_messages(text_collection=search_text_collection, user_query=query)

    if use_dbx_model:
        answer = dbx_llm_chat(
            messages=messages,
            model_name=LLM_model,
            temperature=temperature,
        )
    else:
        # "gpt-5-mini-2025-08-07"
        openai.api_key = load_api_key("gpt_api_key", "config.yaml")
        answer = ask_gpt(
            messages=messages,
            model=LLM_model,
            temperature=temperature,
        )

    return search_df, answer

In [0]:
qa_dict = [
    {
        "question": "How did the 2021 winter storm impact Austin's infrastructure in terms of power and electricity",
        "fact": "Approximately 40% of Austin Energy customers were without power.",
        "fact_source": "2021 Winter Storm Uri After-Action Review",
    },
    {
        "question": "What were the major barriers to effective communication for people with disabilities during Hurricane Harvey",
        "fact": "Barriers included lack of Text-to-911, system overloads, dependence on power, captioning failures, limited interpreter availability, and restricted access to information.",
        "fact_source": "Hurricane Harvey After Action Report on Individuals with Disabilities",
    },
    {
        "question": "What are the four Readiness Levels defined in the Brazos County Emergency Management Plan",
        "fact": "Level 4: Normal Conditions; Level 3: Increased Readiness; Level 2: High Readiness; Level 1: Maximum Readiness (e.g., tropical weather threat, tornado warning, flash flood).",
        "fact_source": "Brazos County Emergency Management Plan",
    },
    {
        "question": "What issues were identified regarding contractor safety during the storm Mara?",
        "fact": "Contractors exhibited unsafe behaviors. Recommended actions include pre-approval of contractor safety plans and formal processes to document, report, and correct unsafe behavior. A related finding noted a defective utility pole previously tagged as unsafe contributed to the Smokehouse Creek Fire.",
        "fact_source": "Winter Storm Mara: After Action Report",
    },
    {
        "question": "What is CRISP program in Austin?",
        "fact": "CRISP stands for the Community Resiliency Improvement Status Portal, developed by the Austin and Travis County Emergency Management Offices to provide transparency into disaster response and recovery efforts.",
        "fact_source": "Austin CRISP",
    },
    {
        "question": "How does the CRISP dashboard track the status of After-Action Report recommendations",
        "fact": "Statuses include Completed (verified finished), In Progress (with percentage completion), Not Started, On Hold or Pending (awaiting assignment or resources), and Closed (cannot be completed or merged into another item).",
        "fact_source": "Austin CRISP",
    },
    {
        "question": "What location was used for a full-scale county-wide POD exercise in July 2017 to simulate recovery from a Category 4 hurricane",
        "fact": "The NRG Center was used for the full-scale POD exercise.",
        "fact_source": "Harris County Tests Points of Distribution Plan",
    },
    {
        "question": "How did Hurricane Harvey flooding impact POD locations in Houston?",
        "fact": "Flooding made many POD sites and routes inaccessible, reduced effectiveness of pre-selected locations, and required post-incident site authorization. Studies indicate flood-aware POD planning could reduce travel distance by 46.5%. Infrastructure failures further constrained viable POD locations.",
        "fact_source": "https://pubmed.ncbi.nlm.nih.gov/32441037/",
    },
    {
        "question": "What specific equipment is required for a Type I POD?",
        "fact": "Required equipment includes 3 forklifts, 3 pallet jacks, 2 power light sets, 6 portable toilets, 2 tents, 4 dumpsters, 30 traffic cones, and 4 two-way radios.",
        "fact_source": "Harris County Points of Distribution (POD) Plan",
    },
    {
        "question": "What are the three core components used to assess flood risk in the 2024 State Flood Plan",
        "fact": "Flood risk is assessed using Flood Hazard (magnitude, location, frequency), Potential Exposure (people and property in hazard areas), and Vulnerability (degree of impact on communities and facilities).",
        "fact_source": "2024 State Flood Plan | Texas Water Development Board",
    },
    {
        "question": "How many Texans currently live in a 100-year floodplain?",
        "fact": "Approximately 2.4 million Texans live or work within the 1 percent (100-year) annual chance floodplain.",
        "fact_source": "2024 State Flood Plan | Texas Water Development Board",
    },
    {
        "question": "What is the Galveston Bay Surge Protection project's total cost?",
        "fact": "The Coastal Texas Project is authorized at $34.38B. The Galveston Bay Storm Surge Barrier System accounts for $31.20B, with a 65% federal share and 35% non-federal share.",
        "fact_source": "2024 State Flood Plan | Texas Water Development Board",
    },
    {
        "question": "How do playa improvements help reduce state flood risk?",
        "fact": "Playa improvements mitigate flat-terrain ponding through region-specific, nature-based and structural flood mitigation solutions.",
        "fact_source": "Technical Guidelines for Regional Flood Planning | TWDB",
    },
    {
        "question": "Tell me more about the strategic buyout program in Onion Creek.",
        "fact": "The voluntary buyout program addresses repeated flooding by purchasing homes along Upper Onion Creek, removing structures, restoring land through regrading and revegetation, and preserving it as permanent open space with potential community benefits.",
        "fact_source": "Nature-based solution Hill Country documentation",
    },
    {
        "question": "How does the flat terrain of the Houston area influence the choice of hydraulic modeling software used for flood planning",
        "fact": "Flat coastal terrain causes widespread, multi-directional flooding requiring 2D hydraulic analysis. Houston-area planning commonly uses HEC-RAS 2D or XPSWMM, with 1D/2D coupling or alternatives needed to represent underground drainage systems.",
        "fact_source": "Technical Guidelines for Regional Flood Planning | TWDB",
    },
]


In [0]:
dbx_llms = [
    # "databricks-claude-opus-4-6",
    # "databricks-claude-sonnet-4-5",
    # "databricks-claude-haiku-4-5",
    # "databricks-claude-opus-4-5",
    "databricks-claude-opus-4-1",
    "databricks-claude-sonnet-4",
    # "databricks-gpt-oss-120b",
    "databricks-gpt-oss-20b",
    "databricks-llama-4-maverick",
    "databricks-gemma-3-12b",
    # "databricks-meta-llama-3-1-8b-instruct",
    # "databricks-meta-llama-3-3-70b-instruct",
]


In [0]:
mlflow.autolog(disable=True)

for LLM_model in tqdm(dbx_llms, desc="LLMs"):
    for item in tqdm(qa_dict, desc=f"QA | {LLM_model}", leave=False):
        query = item["question"]
        try:
            _, answer = RAG_pipe_main(spark, query, LLM_model=LLM_model, temperature=0.2, use_dbx_model=True)
        except Exception:
            _, answer = RAG_pipe_main(spark, query, LLM_model=LLM_model, use_dbx_model=True)
        item[f"answer_{LLM_model}"] = answer

In [0]:
import json
from pyspark.sql import Row

def _to_str(v):
    # Normalize values to avoid Spark type merge errors
    if v is None:
        return None
    if isinstance(v, (str, int, float, bool)):
        return str(v)
    return json.dumps(v, ensure_ascii=False)

def normalize_qa_dict(rows):
    # Normalize keys and values for Spark
    out = []
    for r in rows:
        rr = {}
        for k, v in r.items():
            kk = k.replace("-", "_")  # make Spark-friendly column names
            rr[kk] = _to_str(v)
        out.append(rr)
    return out

qa_rows = normalize_qa_dict(qa_dict)

df = spark.createDataFrame([Row(**x) for x in qa_rows])

table_fqn = "tdis_dev_data_catalog.tdir.qa_dict_eval"
df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(table_fqn)


In [0]:
display(df)